# Create Work Citations (oxjob #2314)

One row per citation: the citing work, the cited work, and every input that holds it (each filed reference list with its
version tier, legacy MAG, or the frozen per-location arrays). Built once per night after `parse_referenced_works` and
`Merged_Work_Ids_Map`; `Works_Enriched` sets each work's `referenced_works` from the rows with `served = true` (sorted
ids, first 5,000).

`served` (oxjob #2313, Casey 17 Nov and 22 Nov 2026): a work whose own lists link anything is served only what a list or
a curated add holds, so legacy MAG and the frozen per-location arrays add nothing to it; a work whose lists link nothing
is served its legacy MAG citations; the frozen arrays alone serve nothing. Reference curations (oxjob #2303) lie on top: a curated add is a source (`curation`), a curated remove sets `served` false, `curation` names the action.
Sized on night 11-22's table: served 3,645,443,414 -> 3,384,988,450 (-260,454,964: 246.5M legacy MAG and 13.9M frozen-array
citations on works whose lists link something or that only the arrays held). ~5 min on shared-2xl.

In [0]:
-- oxjob #2314: one row per citation (citing work, cited work) with every input that holds it, built once per night; it is
-- where Works_Enriched takes each work's referenced_works from. Its inputs: our filed lists (work_references, each list with its
-- version tier: published and accepted 1, submitted 2, other 3), legacy MAG (openalex.mid.citation) and the frozen
-- per-location arrays (openalex_works_base.referenced_works, from location_enrichments). Both ends follow tonight's
-- merged_work_ids_map; no work cites itself; the citing work is one of tonight's works.
-- oxjob #2313 (Casey 22 Nov): a work whose lists link anything is served only what a list or a curated add holds; legacy MAG
-- is served only where a work's lists link nothing; the frozen arrays alone serve nothing (all stay listed in sources).
-- oxjob #2303: reference curations lie on top (work_reference_curations, the latest add or remove per pair; both ends follow
-- merges): an add is one more source of the pair ('curation'; the cited work must be one of tonight's works), a remove
-- sets served false. The inputs stay listed in sources, so deleting the curation serves them again the next night.
CREATE OR REPLACE TABLE openalex.works.work_citations
CLUSTER BY (citing_work_id)
TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '7 days', 'delta.logRetentionDuration' = '7 days')
AS
WITH m AS (SELECT id AS k, merge_into_id AS v FROM openalex.works.merged_work_ids_map),
lists AS (
  SELECT l.native_id, l.native_id_namespace, l.provenance, coalesce(mc.v, l.citing_work_id) AS citing_work_id,
    CASE WHEN l.version_rank <= 2 THEN 1 WHEN l.version_rank = 3 THEN 2 ELSE 3 END AS version_tier
  FROM openalex.works.work_reference_lists_current l
  LEFT JOIN m mc ON mc.k = l.citing_work_id
),
best AS (SELECT citing_work_id, min(version_tier) AS best_tier FROM lists GROUP BY citing_work_id),
cur AS (
  SELECT coalesce(mc.v, c.work_id) AS citing_work_id, coalesce(md.v, c.cited_work_id) AS cited_work_id, c.action
  FROM openalex.works.work_reference_curations c
  LEFT JOIN m mc ON mc.k = c.work_id
  LEFT JOIN m md ON md.k = c.cited_work_id
  QUALIFY ROW_NUMBER() OVER (PARTITION BY coalesce(mc.v, c.work_id), coalesce(md.v, c.cited_work_id)
                             ORDER BY c.created DESC, c.curation_id DESC) = 1
),
edges AS (
  SELECT coalesce(mc.v, wr.citing_work_id) AS citing_work_id, coalesce(md.v, wr.cited_work_id) AS cited_work_id,
    named_struct('provenance', wr.provenance, 'native_id_namespace', wr.native_id_namespace, 'native_id', wr.native_id,
                 'version_tier', l.version_tier) AS source
  FROM openalex.works.work_references wr
  LEFT JOIN lists l
    ON l.native_id = wr.native_id AND l.native_id_namespace = wr.native_id_namespace AND l.provenance = wr.provenance
  LEFT JOIN m mc ON mc.k = wr.citing_work_id
  LEFT JOIN m md ON md.k = wr.cited_work_id
  WHERE wr.cited_work_id IS NOT NULL
  UNION ALL
  SELECT coalesce(mc.v, c.paper_id), coalesce(md.v, c.paper_reference_id),
    named_struct('provenance', 'legacy-mag', 'native_id_namespace', CAST(NULL AS STRING), 'native_id', CAST(NULL AS STRING),
                 'version_tier', CAST(NULL AS INT))
  FROM openalex.mid.citation c
  LEFT JOIN m mc ON mc.k = c.paper_id
  LEFT JOIN m md ON md.k = c.paper_reference_id
  UNION ALL
  SELECT x.id, coalesce(md.v, x.cited),
    named_struct('provenance', 'frozen-array', 'native_id_namespace', CAST(NULL AS STRING), 'native_id', CAST(NULL AS STRING),
                 'version_tier', CAST(NULL AS INT))
  FROM (SELECT id, explode(referenced_works) AS cited FROM openalex.works.openalex_works_base WHERE size(referenced_works) > 0) x
  LEFT JOIN m md ON md.k = x.cited
  UNION ALL
  SELECT cur.citing_work_id, cur.cited_work_id,
    named_struct('provenance', 'curation', 'native_id_namespace', CAST(NULL AS STRING), 'native_id', CAST(NULL AS STRING),
                 'version_tier', CAST(NULL AS INT))
  FROM cur LEFT SEMI JOIN openalex.works.openalex_works_base w ON w.id = cur.cited_work_id
  WHERE cur.action = 'add'
),
pairs AS (
  SELECT citing_work_id, cited_work_id, array_sort(collect_set(source)) AS sources
  FROM edges
  WHERE citing_work_id IS NOT NULL AND cited_work_id IS NOT NULL AND citing_work_id <> cited_work_id
  GROUP BY citing_work_id, cited_work_id
),
listed AS (
  SELECT DISTINCT citing_work_id FROM pairs
  WHERE exists(sources, s -> s.provenance NOT IN ('legacy-mag', 'frozen-array', 'curation'))
)
SELECT p.citing_work_id, p.cited_work_id, p.sources,
  coalesce(exists(p.sources, s -> s.version_tier = b.best_tier), FALSE) AS in_best_version,
  exists(p.sources, s -> s.provenance = 'legacy-mag') AS from_legacy_mag,
  exists(p.sources, s -> s.provenance = 'frozen-array') AS from_frozen_arrays,
  CASE WHEN cur.action = 'remove' THEN FALSE
       WHEN exists(p.sources, s -> s.provenance NOT IN ('legacy-mag', 'frozen-array')) THEN TRUE
       ELSE lw.citing_work_id IS NULL AND exists(p.sources, s -> s.provenance = 'legacy-mag') END AS served,
  cur.action AS curation,
  current_timestamp() AS built_at
FROM pairs p
LEFT JOIN best b ON b.citing_work_id = p.citing_work_id
LEFT JOIN cur ON cur.citing_work_id = p.citing_work_id AND cur.cited_work_id = p.cited_work_id
LEFT JOIN listed lw ON lw.citing_work_id = p.citing_work_id
LEFT SEMI JOIN openalex.works.openalex_works_base w ON w.id = p.citing_work_id
